# Extracting precipitation data from CHIRPS

In [ ]:
import pandas as pd
import requests
from pathlib import Path
import xarray as xr
import numpy as np


In [26]:
lonlat = pd.read_csv('../data/raw/locations.csv')

In [ ]:
""""
base_url = (
    "https://data.chc.ucsb.edu/products/"
    "CHIRPS-2.0/global_daily/netcdf/p05"
)

years = range(2020, 2026)

out_dir = Path("../data/raw/chirps")
out_dir.mkdir(parents=True, exist_ok=True)

for y in years:
    url = f"{base_url}/chirps-v2.0.{y}.days_p05.nc" # Daily precipitation totals
    out_file = out_dir / f"chirps_{y}.nc"

    if out_file.exists():
        print(f"{y} already downloaded")
        continue

    print(f"Downloading {y}...")
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()

    with open(out_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
"""

In [ ]:
files = sorted(Path("../data/raw/chirps").glob("chirps_*.nc"))

chirps = xr.open_mfdataset(
    files,
    engine="netcdf4",
    combine="by_coords",
    parallel=False
)

In [ ]:
points = lonlat.reset_index(drop=True)

lons = xr.DataArray(
    points.lon.values,
    dims="points"
)

lats = xr.DataArray(
    points.lat.values,
    dims="points"
)

In [29]:
precip = chirps["precip"].sel(
    longitude=lons,
    latitude=lats,
    method="nearest"
)


In [30]:
weekly = precip.resample(time="1W")

weekly_stats = xr.Dataset({
    "prep_week_min": weekly.min(),
    "prep_week_max": weekly.max(),
    "prep_week_mean": weekly.mean(),
    "prep_week_median": weekly.median(),
    "prep_week_sum": weekly.sum()
})

datos_weekly = (
    weekly_stats
    .to_dataframe()
    .reset_index()
    .assign(
        lon=lambda df: points.loc[df["points"], "lon"].values,
        lat=lambda df: points.loc[df["points"], "lat"].values,
        canton=lambda df: points.loc[df["points"], "canton"].values
    )
    .drop(columns="points")
    .rename(columns={"time": "week"})
)

In [31]:
datos_weekly.to_csv('../data/raw/chirps_precip_weekly.csv', index=False)